In [ ]:
!pip install humancompatible-train

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 807.7 kB/s eta 0:00:00


# To start


In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 48.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/Causality/Recommender System-DNN/Andrii-hcomp-lite/")

# !export PYTHONPATH="$PYTHONPATH:/content/drive/MyDrive/Colab Notebooks/NCPOP-Colab Notebooks-data"
import os
os.environ['GRB_LICENSE_FILE'] = '/content/drive/MyDrive/gurobi.lic'  # Update this path with the actual license location


Mounted at /content/drive


In [ ]:
import sys
from humancompatible.train.dual_optim import ALM, MoreauEnvelope, PBM
import torch
from torch import nn
import numpy as np


**Define regressor**

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, lsize):
        super().__init__()
        layers = []
        d = input_dim
        for dim in lsize:
            layers.append(nn.Linear(d, dim))
            layers.append(nn.Dropout(p=0.15))
            layers.append(nn.ReLU())
            d = dim

        layers.append(nn.Linear(d, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def predict(self, X):
        return self.net(torch.tensor(X, dtype=torch.float32)).squeeze(-1)

**Prepare data**

Load config (used for solving DAG problem, setting target, etc)

In [ ]:
from omegaconf import OmegaConf
import os

# Ensure the folder 'experiments_conf' exists at this location.
config_path = "experiments_conf/solver/hc_predictor.yaml"
cfg = OmegaConf.load(config_path)

problem_config_path = "experiments_conf/problem/codiet.yaml"
# This line failed because the file was not found at the path above.
problem_cfg = OmegaConf.load(problem_config_path)

Load data

In [ ]:
import os
from data_helper import load_all_data
import pandas as pd

hydra_run_path = "codiet-select"
df_features = pd.read_feather(f"{hydra_run_path}/features.feather")
selected_features = df_features.columns.tolist()

target_col = problem_cfg.target
food_feats, non_food_feats, prep_data = load_all_data()

# Only keep features that actually exist in the loaded data to prevent KeyError
available_features = [feat for feat in selected_features if feat in prep_data.columns]
prep_data = prep_data[available_features + [target_col]]

w_path = "codiet-select/W_est_02.csv"
with open(w_path) as f:
    W = pd.read_csv(f, delimiter=",", header=None)

scafs: Dropping cols with many nans: ['scafs_aliquot-no', 'scafs_volume-per-aliqote', 'scafs_day', 'scafs_age', 'scafs_gender', 'scafs_box-position']. 
scafs: (282, 13)
ms_urine: (300, 36)
ms_serum: Dropping cols with many nans: ['ms_serum_aminoadipic-acid', 'ms_serum_beta-alanine', 'ms_serum_gamma-aminobutyric-acid']. 
ms_serum: (300, 33)
nmr_urine: (298, 53)
ms_lip: (301, 46)
dbs_rbc_lip: (299, 60)
microbiome: prep shape: (151, 4)
scafs: prep shape: (149, 5)
ms_serum: prep shape: (154, 30)
ms_urine: prep shape: (154, 33)
nmr_urine: prep shape: (153, 50)
lipidomics: prep shape: (155, 45)
lipidomics_dbs_rbc: prep shape: (153, 59)
microbiome_4_cl: prep shape: (151, 5)
microbiome_phyl4_cl: prep shape: (151, 5)
microbiome_embedding: prep shape: (151, 21)
microbiome_clean15: prep shape: (151, 16)

Total prep_data shape: (140, 488)
0      1.0
1      1.0
2      1.0
3      1.0
4      0.0
      ... 
135    0.0
136    1.0
137    1.0
138    1.0
139    1.0
Name: gender_numeric, Length: 140, dtype

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize

def p_dweibull(y, q_alpha, beta):
    cdf = 1.0 - np.power(q_alpha, np.power(y + 1, beta))
    cdf[y < 0] = 0.0
    return cdf

def to_gaussian_table(X, alpha, beta):
    columns = X.columns if isinstance(X, pd.DataFrame) else None
    index = X.index if isinstance(X, pd.DataFrame) else None

    X_arr = np.asarray(X, dtype=np.float64)
    n, p = X_arr.shape

    Z_mid = np.zeros((n, p))
    Z_rand = np.zeros((n, p))

    for j in range(p):
        y = X_arr[:, j]

        Fy = p_dweibull(y, alpha[j], beta[j])
        Fy_ = p_dweibull(y - 1, alpha[j], beta[j])

        U_mid = (Fy + Fy_) / 2.0
        U_mid = np.clip(U_mid, 1e-15, 1 - 1e-15)
        Z_mid[:, j] = norm.ppf(U_mid)

        U_rand = np.random.uniform(Fy_, Fy, size=n)
        U_rand = np.clip(U_rand, 1e-15, 1 - 1e-15)
        Z_rand[:, j] = norm.ppf(U_rand)

    if columns is not None:
        Z_mid = pd.DataFrame(Z_mid, index=index, columns=columns)
        Z_rand = pd.DataFrame(Z_rand, index=index, columns=columns)

    return {"Z_mid": Z_mid, "Z_rand": Z_rand}

def estimate_dweibull_params(X):
    X_arr = np.asarray(X, dtype=np.float64)
    p = X_arr.shape[1]

    alpha_list = []
    beta_list = []

    for j in range(p):
        y = X_arr[:, j]

        def nll(params):
            a, b = params
            if a <= 0 or a >= 1 or b <= 0:
                return 1e10

            term1 = np.power(a, np.power(y, b))
            term2 = np.power(a, np.power(y + 1, b))
            pmf = term1 - term2

            pmf = np.clip(pmf, 1e-12, 1.0)
            return -np.sum(np.log(pmf))

        initial_guess = [0.5, 1.0]
        bounds = [(1e-5, 1 - 1e-5), (1e-5, 50.0)]

        res = minimize(nll, initial_guess, bounds=bounds, method='L-BFGS-B')

        if res.success:
            alpha_list.append(res.x[0])
            beta_list.append(res.x[1])
        else:
            alpha_list.append(0.5)
            beta_list.append(1.0)

    return np.array(alpha_list), np.array(beta_list)


In [ ]:
# 把离散 Weibull 边缘分布数据转为连续高斯分布
alpha, beta = estimate_dweibull_params(X)
gaussian_tables = to_gaussian_table(X, alpha, beta)

prep_data = prep_data.dropna(subset=[target_col])
prep_data = prep_data.dropna(axis=1)
X = prep_data.drop(target_col, axis=1) if target_col in prep_data.columns else prep_data
y = prep_data[target_col]
X = X.to_numpy()
y = y.to_numpy()

Train:

In [ ]:
# network params
lsize = [32, 16]

# training method params
lr = 0.25
init_lambda = 1.
lambda_lr = 1.0
penalty = 0.

sample_weight = None

# training process params
n_outer = 10
n_inner = 100
verbose = True

torch.manual_seed(42)

## SPBM


SPBM via `humancompatible.train.dual_optim.PBM` (paper Algorithm 1)


In [ ]:
import networkx as nx
%run solve_milp.py
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from humancompatible.train.dual_optim import PBM, MoreauEnvelope


def spbm_w_constraints(W, X_batch, yhat, tolerance=1e-3):
    """Convert the W/DAG equality into inequalities accepted by PBM."""
    d = X_batch.shape[1]
    M = W - torch.eye(d + 1, device=W.device, dtype=W.dtype)
    xy_bar = torch.cat([
        X_batch.mean(dim=0),
        yhat.reshape(-1).mean().reshape(1),
    ])
    residual = M @ xy_bar
    return residual.abs() - tolerance


def fit_nn_estimator_spbm(
    X,
    y,
    W,
    model,
    verbose=False,
    constrained=True,
    total_steps=500,
    batch_size=128,
    learning_rate=1e-3,
    gamma=0.9,
    prox_mu=1.0,
    center_decay=0.5,
    penalty_mult=0.9,
    tolerance=1e-3,
):
    """Train with the PBM implementation corresponding to SPBM Algorithm 1."""
    _, d = X.shape
    assert W.shape == (d + 1, d + 1), "W must be (d+1)x(d+1)"

    device = next(model.parameters()).device
    X = X.to(device=device, dtype=torch.float32)
    y = y.to(device=device, dtype=torch.float32).reshape(-1)
    W = W.to(device=device, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X, y),
        batch_size=min(batch_size, len(X)),
        shuffle=True,
        drop_last=False,
    )
    loader_iter = iter(loader)

    base_opt = torch.optim.Adam(model.parameters(), lr=learning_rate)
    primal_opt = (
        MoreauEnvelope(
            base_opt,
            mu=prox_mu,
            beta=1.0 - center_decay,
        )
        if constrained
        else base_opt
    )

    dual_opt = None
    if constrained:
        dual_opt = PBM(
            m=d + 1,
            penalty_mult=penalty_mult,
            gamma=gamma,
            delta=1.0,
            penalty_update="dimin_adapt",
            pbf="quadratic_logarithmic",
            init_duals=0.1,
            init_penalties=1.0,
            dual_range=(1e-4, 100.0),
            penalty_range=(0.1, 1.0),
            primal_update_process_length=1,
            device=device,
        )

    loss_fn = torch.nn.MSELoss()

    for step in range(total_steps):
        try:
            X_batch, y_batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            X_batch, y_batch = next(loader_iter)

        # Call the underlying optimizer directly; this avoids wrapper-version
        # differences in delegated zero_grad implementations.
        base_opt.zero_grad(set_to_none=True)

        yhat = model(X_batch).reshape(-1)
        mse = loss_fn(yhat, y_batch)

        if constrained:
            g = spbm_w_constraints(
                W,
                X_batch,
                yhat,
                tolerance=tolerance,
            )
            total_loss = dual_opt.forward_update(mse, g)
        else:
            g = torch.zeros(d + 1, device=device)
            total_loss = mse

        total_loss.backward()
        primal_opt.step()

        if verbose and step % 50 == 0:
            dual_norm = dual_opt.duals.norm().item() if dual_opt is not None else 0.0
            p_min = dual_opt.penalties.min().item() if dual_opt is not None else 0.0
            p_max = dual_opt.penalties.max().item() if dual_opt is not None else 0.0
            print(
                f"step={step:04d}  "
                f"mean(yhat)={yhat.mean().item():+.6e}  "
                f"MSE={mse.item():.6e}  "
                f"max(g)={g.max().item():+.6e}  "
                f"||g+||={torch.relu(g).norm().item():.6e}  "
                f"lambda_norm={dual_norm:.6e}  "
                f"p=[{p_min:.3e}, {p_max:.3e}]"
            )

    if dual_opt is not None:
        model.spbm_duals_ = dual_opt.duals.detach().cpu()
        model.spbm_penalties_ = dual_opt.penalties.detach().cpu()

    return model


def get_current_column_names(X):
    current_feature_names = []
    for i in range(X.shape[1]):
        col_data = X[:, i]
        for col_name in prep_data.columns:
            if col_name in current_feature_names:
                continue
            it = iter(prep_data[col_name])
            if all(any(a == b for a in it) for b in col_data):
                current_feature_names.append(col_name)
                break
    return current_feature_names


def calculate_dag(X, y):
    d = X.shape[1] + 1
    X_y = np.column_stack((X, y))
    current_column_names = get_current_column_names(X)
    G = nx.read_graphml(os.path.join(cfg.data_path, cfg.knowledge_graph_filename))
    H = G.subgraph(current_column_names + [target_col]).copy()
    if H.number_of_nodes() > 0:
        print("not empty")
    H = nx.complement(H)
    col_to_idx = {
        col: idx
        for idx, col in enumerate(current_column_names + [target_col])
    }
    tabu_edges = [
        (col_to_idx[source], col_to_idx[target])
        for source, target in H.edges()
    ]
    w_est, _, _, _, _ = solve(
        X_y,
        cfg,
        cfg.nonzero_threshold,
        Y=[],
        B_ref=np.zeros((d, d)),
        tabu_edges=tabu_edges,
    )
    return w_est


## ALM


ALM


In [ ]:
import networkx as nx
%run solve_milp.py

def W_constraint(M, muX, yhat):
    muY = yhat.mean()
    xy_bar = torch.concat([muX, muY.unsqueeze(0)])
    g = torch.abs(xy_bar - M @ xy_bar)
    return g

def fit_nn_estimator_alm(X, y, W, model, verbose=False, constrained=True):

    n, d = X.shape

    assert W.shape == (d + 1, d + 1), \
    "W must be (d+1)x(d+1)"

    base_opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.25)
    optimizer = MoreauEnvelope(base_opt) if constrained else base_opt
    dual_opt = ALM(m=d+1, lr=lambda_lr, penalty=penalty, init_duals=init_lambda)

    # ------------------------------
    # Precompute parts of constraint
    # ------------------------------
    M = W - torch.eye(d + 1)
    muX = X.mean(dim=0)
    loss = torch.nn.MSELoss()

    for outer in range(n_outer):
        for inner in range(n_inner):
            optimizer.zero_grad()
            yhat = model(X)
            mse = loss(yhat, y)

            if constrained:
                g = W_constraint(M, muX, yhat)
                aug_loss = dual_opt.forward(loss=mse, constraints=g)
                aug_loss.backward()
            else:
                mse.backward()

            optimizer.step()

        if constrained:
            with torch.no_grad():
                muY = model(X).mean()
                xy_bar = torch.concat([muX, muY.unsqueeze(0)])
                g = torch.abs(xy_bar - M @ xy_bar)
                dual_opt.update(g)
            lam = dual_opt.duals.detach().numpy()

        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.95

        if verbose:
            print(
            f"outer={outer:02d}  "
            f"mean(yhat)={muY.item():+.6e}  "
            f"MSE={mse.item():+.6e}  "
            f"||g||={np.linalg.norm(g).item():.6e}  "
            f"lambda_norm={np.linalg.norm(lam).item():.6e}  "
        )


def get_current_column_names(X):
        current_feature_names = []

        for i in range(X.shape[1]):
            col_data = X[:, i]
            for col_name in prep_data.columns:
                if col_name in current_feature_names:
                    continue
                it = iter(prep_data[col_name])
                if all(any(a == b for a in it) for b in col_data):
                    current_feature_names.append(col_name)
                    break

        return current_feature_names

def calculate_dag(X, y):
    d = X.shape[1] + 1 # adding one for y
    X_y = np.column_stack((X, y))
    current_column_names = get_current_column_names(X)
    G = nx.read_graphml(os.path.join(cfg.data_path, cfg.knowledge_graph_filename))
    H = G.subgraph(current_column_names + [target_col]).copy()
    if H.number_of_nodes() > 0:
        print('not emty')
    H = nx.complement(H)
    col_to_idx = {col: idx for idx, col in enumerate(current_column_names + [target_col])}
    tabu_edges = list((col_to_idx[s],col_to_idx[e]) for (s,e) in H.edges())
    w_est, _, _, _, _ = solve(X_y, cfg, cfg.nonzero_threshold,
                                                            Y=[],
                                                            B_ref=np.zeros((d,d)),
                                                            tabu_edges=tabu_edges )
    return w_est

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.base import BaseEstimator
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
%run solve_milp.py


class ScikitModelWrapper(BaseEstimator):
    def __init__(
        self,
        W=None,
        constrained=True,
        recalculate_dag=True,
        method="spbm",
    ):
        self.W = W
        self.scaler = StandardScaler()
        self.recalculate_dag = recalculate_dag
        self.constrained = constrained
        self.method = method
        self.is_fitted_ = False

    def fit(self, X, y):
        if self.recalculate_dag:
            self.W = calculate_dag(X, y)
        elif self.W is None:
            raise ValueError(
                f"recalculate_dag set to {self.recalculate_dag} but W not provided!"
            )

        self.model = MLPRegressor(input_dim=X.shape[1], lsize=lsize)
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        W_tensor = torch.tensor(np.asarray(self.W), dtype=torch.float32)

        if self.method == "spbm":
            trainer = fit_nn_estimator_spbm
        elif self.method == "alm":
            trainer = fit_nn_estimator_alm
        else:
            raise ValueError("method must be 'spbm' or 'alm'")

        trainer(
            X_tensor,
            y_tensor,
            W_tensor,
            self.model,
            constrained=self.constrained,
        )
        self.is_fitted_ = True
        return self

    @torch.inference_mode()
    def predict(self, X):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        return self.model(X_tensor).detach().numpy()


@torch.inference_mode()
def compute_predictor_errors_scikit(estimator, X, y):
    return mean_squared_error(y, estimator.predict(X))


# Explicitly select SPBM so executing the ALM section does not override it.
pipe = make_pipeline(
    StandardScaler(),
    ScikitModelWrapper(
        W=None,
        constrained=True,
        recalculate_dag=True,
        method="spbm",
    ),
)

results = cross_validate(
    pipe,
    X,
    y,
    cv=10,
    scoring=compute_predictor_errors_scikit,
    return_train_score=True,
)


In [ ]:
results['train_score'].mean() #/ score_normalizer

np.float64(7.780891654556925)

In [ ]:
results['test_score'].mean() #/ score_normalizer

np.float64(8.472174040015824)

## XGBoost

Try XGBoost on same data:

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline


n_estimators = 10
max_depth = 3
learning_rate = 0.1
random_state = 42


model_class = Pipeline
model_params = {
    'steps': [
        ("scale", StandardScaler()),
        ("xgb", XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=random_state,
        )
        )
    ]
}

xgboost_model = model_class(**model_params)

In [ ]:
# from sklearn.model_selection import GridSearchCV


# gridsearch = GridSearchCV(
#     estimator=Pipeline((('scaler', StandardScaler()), ('xgb', XGBRegressor()))),
#     param_grid={'xgb__n_estimators': [3, 5, 10, 15, 20], 'xgb__max_depth': [1, 2, 3, 4, 5], 'xgb__learning_rate': [0.15, 0.2, 0.25]},
#     scoring=lambda e, X, y: -1*compute_predictor_errors_scikit(e, X, y),
#     return_train_score=True,
#     cv=10,
# )

# gridsearch.fit(X, y)
# gridsearch.best_score_

In [ ]:
results = cross_validate(
    xgboost_model,
    X,
    y,
    cv = 10,
    scoring=compute_predictor_errors_scikit,
    return_train_score=True,
)

In [ ]:
results['train_score'].mean() #/ score_normalizer

np.float64(4.731357115912838)

In [ ]:
results['test_score'].mean() #/ score_normalizer

np.float64(8.251131722940288)